In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║   🌈✨  M A R I A  — Magical AI Reading & Instruction Assistant ✨🌈   ║
# ║              📚 EdTech Adventure Platform for Kids 6-12 📚             ║
# ║                  🦄 Learn · Laugh · Grow with MARIA! 🦄                ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# ⚡ HOW TO USE THIS NOTEBOOK IN GOOGLE COLAB:
#   1️⃣  Runtime → Change runtime type → Select GPU (T4 recommended)
#   2️⃣  Run Cell 1: Install tools
#   3️⃣  Run Cell 2: Download files
#   4️⃣  Run Cell 3: Load MARIA's brain (takes ~5 min)
#   5️⃣  Run Cell 4: Helper functions
#   6️⃣  Run Cell 5: Launch MARIA! 🚀
#
# ==============================================================================


# %%
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🛠️  CELL 1 — Install All The Magic Tools!  (Run This First!)  ║
# ╚══════════════════════════════════════════════════════════════════╝

import subprocess, sys

print("""
🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈
  ✨ Welcome to MARIA Setup! Installing magic tools... ✨
🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈
""")

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "gradio>=4.40.0",
    "transformers>=4.45.0",
    "bitsandbytes>=0.43.0",
    "accelerate>=0.32.0",
    "faiss-cpu",
    "pandas",
    "pyarrow",
    "sentence-transformers",
    "huggingface_hub",
    "requests",
    "diffusers>=0.28.0",   # TextToVideoSDPipeline (ZeroScope V2) requires >=0.28
    "imageio",
    "imageio-ffmpeg",      # export_to_video needs ffmpeg backend
    "Pillow",
], check=True)

print("""
🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊
  🎉 ALL TOOLS INSTALLED SUCCESSFULLY! 🎉
  🚀 MARIA is almost ready to teach! 🚀
🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊🎊
""")


# %%
# ╔═══════════════════════════════════════════════════════════════╗
# ║  📥  CELL 2 — Download Files & Prepare MARIA's Knowledge!   ║
# ╚═══════════════════════════════════════════════════════════════╝

import os
import json
import zipfile
import requests
from io import BytesIO

BASE_DIR          = "/content"
KNOWLEDGEBASE_DIR = f"{BASE_DIR}/knowledgebase"
CURRICULUM_DIR    = f"{BASE_DIR}/curriculum"

os.makedirs(KNOWLEDGEBASE_DIR, exist_ok=True)
os.makedirs(CURRICULUM_DIR,    exist_ok=True)

# ── Helpers ──────────────────────────────────────────────────────────────────

def download_and_extract(url: str, extract_to: str, label: str) -> None:
    """Stream-download a ZIP and extract it, showing progress."""
    print(f"  📥 Downloading {label} …")
    resp = requests.get(url, stream=True, timeout=120)
    resp.raise_for_status()
    total   = int(resp.headers.get("content-length", 0))
    buffer  = BytesIO()
    fetched = 0
    for chunk in resp.iter_content(chunk_size=32_768):
        buffer.write(chunk)
        fetched += len(chunk)
        if total:
            pct = fetched / total * 100
            bar = "█" * int(pct // 5) + "░" * (20 - int(pct // 5))
            print(f"\r  [{bar}] {pct:5.1f}%", end="", flush=True)
    print(f"\r  ✅ Downloaded {label}!{' ' * 30}")
    buffer.seek(0)
    with zipfile.ZipFile(buffer) as zf:
        zf.extractall(extract_to)
    print(f"  📂 Extracted to  →  {extract_to}\n")


def fetch_json(url: str, label: str) -> dict | None:
    """Download and parse JSON from a URL."""
    print(f"  📥 Downloading {label} …")
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    print(f"  ✅ Got {label}!\n")
    return resp.json()


# ── Run downloads ─────────────────────────────────────────────────────────────

print("\n🌟 ═══ Starting Downloads ═══ 🌟\n")

download_and_extract(
    "https://raw.githubusercontent.com/buildwithsupratim/maria/main/knowledgebase.zip",
    BASE_DIR,
    "🧠 Knowledge Base (RAG)"
)

download_and_extract(
    "https://raw.githubusercontent.com/buildwithsupratim/maria/main/curriculum.zip",
    BASE_DIR,
    "📚 Curriculum"
)

TEACHER_PERSONAS = fetch_json(
    "https://raw.githubusercontent.com/buildwithsupratim/maria/refs/heads/main/teacherpersona.json",
    "👩‍🏫 Teacher Personas"
)

print("""
🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉
  ✅ All downloads done! MARIA's knowledge is ready!
🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉
""")


# %%
# ╔═════════════════════════════════════════════════════════════════════╗
# ║  🤖  CELL 3 — Load MARIA's Brain (Qwen3-4B INT4 + Embeddings)!   ║
# ║       ☕ Grab a snack — this takes about 5 minutes!  ☕           ║
# ╚═════════════════════════════════════════════════════════════════════╝

import re
import random
import warnings
import numpy  as np
import pandas as pd
import faiss
import torch
from transformers          import (AutoModelForCausalLM, AutoTokenizer,
                                   BitsAndBytesConfig)
from sentence_transformers import SentenceTransformer
import gradio              as gr

warnings.filterwarnings("ignore")

# ── Constants ─────────────────────────────────────────────────────────────────

MODEL_NAME = "Qwen/Qwen3-4B"

TYPING_MESSAGES = [
    "🎵 Gretl, if you hide my guitar one more time, I'm enrolling you in extra singing practice with Kurt! 🎵",
    "🏃 Kurt, stop running! Education is chasing you faster than I am! 🏃",
    "📖 Liesl, you're the eldest — kindly use your powers for good, not for helping Friedrich escape math! 📖",
    "✏️ Friedrich, that's not 'creative learning,' that's drawing mustaches on your homework! ✏️",
    "🦋 Louisa, I admire your imagination, but convincing Marta that books bite is not acceptable! 🦋",
    "📢 Brigitta, thank you for telling the truth… but did you have to announce it like a town crier? 📢",
    "⚡ Marta, if you can run this fast during playtime, you can run just as fast to your lessons! ⚡",
    "🐴 Gretl, no, Maria is not your horse. Please dismount immediately! 🐴",
    "🧠 Kurt, I don't care how fast you are — knowledge will catch you eventually! 🧠",
    "🎶 All of you, line up! If we can sing in harmony, we can definitely sit quietly for five minutes… right? 🎶",
    "🐸 Friedrich, if you put that frog in my pocket again, it will be attending your next lesson as your study partner! 🐸",
    "🗺️ Louisa, hiding the lesson plan does not cancel the lesson — it just makes me more creative! 🗺️",
    "🦵 Kurt, I admire your speed, but unfortunately for you, I also have determination and longer legs! 🦵",
    "🚪 Liesl, as the eldest, you are supposed to set an example — not set up escape routes! 🚪",
    "😲 Brigitta, honesty is wonderful, but announcing 'Maria is about to be tricked again' ruins the suspense! 😲",
    "🎉 Marta, you cannot declare a 'national holiday' just because you don't feel like studying today! 🎉",
    "🪄 Gretl, that is not a 'magic wand,' that is my spoon — please stop casting spells with it! 🪄",
    "🎨 Friedrich, if your homework has more doodles than answers, I will start grading the doodles! 🎨",
    "🤝 Louisa, teamwork is lovely, but teaming up against your governess was not part of today's lesson! 🤝",
    "🤫 All of you, if silence were a subject, you would all currently be failing spectacularly! 🤫",
]

# ── ZeroScope V2 progress messages (shown when generating visual videos) ──────

SD_TYPING_MESSAGES = [
    "🎬 Warming up the ZeroScope V2 video engine… ✨",
    "🖼️ Sketching keyframes for your learning video… ✏️",
    "🌈 ZeroScope is weaving magical motion pixels… 🎭",
    "⚡ Rendering frames at full speed — almost there! 🚀",
    "🦄 Adding sparkles and finishing touches… 🌟",
    "🔮 Your video explanation is crystallising! 💎",
    "🐝 The animation-bees are buzzing furiously… 🍯",
    "🌟 One moment — finalising your learning clip! 🎬",
]

# ── ZeroScope V2 video generation helper ─────────────────────────────────────
# Model: cerspense/zeroscope_v2_576w
# • Native diffusers TextToVideoSDPipeline — has model_index.json ✅
# • ~3.9 GB fp16 weights — coexists with Qwen3-4B INT4 on T4 with CPU offload ✅
# • 576×320 native resolution, 24 frames @ 8 fps = 3-second clips ✅

_VIDEO_PATH = "/tmp/maria_visual_aid.mp4"

def generate_visual_video(prompt: str) -> str | None:
    """
    Generate a colourful, kid-safe educational short video using ZeroScope V2.
    Returns the saved .mp4 path, or None on failure.
    """
    try:
        from diffusers.utils import export_to_video
        safe_prompt = (
            "Educational animation for children ages 6-12, bright colours, "
            "friendly cartoon style, simple and clear, safe for kids, "
            f"{prompt}"
        )
        output = wan_pipe(
            prompt          = safe_prompt,
            num_frames      = 24,       # 3 s @ 8 fps — fast enough on T4
            width           = 576,      # ZeroScope native width
            height          = 320,      # ZeroScope native height
            num_inference_steps = 25,
            guidance_scale  = 7.5,
        )
        # output.frames[0] → list of PIL Images (one per frame, first video in batch)
        export_to_video(output.frames[0], _VIDEO_PATH, fps=8)
        return _VIDEO_PATH
    except Exception as exc:
        print(f"⚠️  ZeroScope video generation failed: {exc}")
        return None

# ── Load Qwen3-4B with INT4 quantisation ─────────────────────────────────────

print(f"\n🤖 Loading {MODEL_NAME} with INT4 quantisation …\n")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.eval()
print("✅ Qwen3-4B loaded!\n")

# ── Load ZeroScope V2 for kid-friendly visual learning ───────────────────────
# cerspense/zeroscope_v2_576w is a fully diffusers-native TextToVideoSDPipeline.
# It ships with model_index.json so from_pretrained works reliably.
# CPU offload keeps peak VRAM ≈ 4 GB, leaving headroom for Qwen3-4B INT4.

print("🎬 Loading ZeroScope V2 (cerspense/zeroscope_v2_576w) for visual video generation …")
from diffusers import TextToVideoSDPipeline

wan_pipe = TextToVideoSDPipeline.from_pretrained(
    "cerspense/zeroscope_v2_576w",
    torch_dtype=torch.float16,
)
wan_pipe.enable_model_cpu_offload()   # stream layers to CPU; keeps T4 VRAM free
wan_pipe.enable_vae_slicing()         # further reduce peak VRAM during decode
print("✅ ZeroScope V2 loaded!\n")

# ── Load sentence-transformer for RAG ────────────────────────────────────────

print("🔍 Loading embedding model for RAG …")
EMBED_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("✅ Embedding model ready!\n")

print("""
🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆
  🧠 MARIA's Brain is FULLY LOADED and ready to teach! 🧠
🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆🏆
""")


# %%
# ╔═══════════════════════════════════════════════════════════════╗
# ║  🧩  CELL 4 — Helper Functions (RAG · Decision · Inference)  ║
# ╚═══════════════════════════════════════════════════════════════╝

_curriculum_cache: dict = {
    "boards":     [],
    "classes":    {},
    "subjects":   {},
    "curriculum": {},
}


def get_boards() -> list[str]:
    return _curriculum_cache["boards"]


def get_classes(board: str) -> list[str]:
    return _curriculum_cache["classes"].get(board, [])


def get_subjects(board: str, cls: str) -> list[str]:
    return _curriculum_cache["subjects"].get(f"{board}/{cls}", [])


def get_curriculum(board: str, cls: str, subject: str) -> list[dict]:
    return _curriculum_cache["curriculum"].get(f"{board}/{cls}/{subject}", [])


# ── FAISS / RAG helpers ───────────────────────────────────────────────────────

_faiss_cache: dict = {}

def load_faiss(board: str, cls: str, subject: str):
    key = f"{board}/{cls}/{subject}"
    if key in _faiss_cache:
        return _faiss_cache[key]

    base        = os.path.join(KNOWLEDGEBASE_DIR, board, cls, subject)
    bin_path    = os.path.join(base, "faiss_index.bin")
    parq_path   = os.path.join(base, "metadata.parquet")
    cfg_path    = os.path.join(base, "config.json")

    if not os.path.exists(bin_path):
        return None, None, {}

    index    = faiss.read_index(bin_path)
    metadata = pd.read_parquet(parq_path) if os.path.exists(parq_path) else pd.DataFrame()
    config   = json.load(open(cfg_path)) if os.path.exists(cfg_path) else {}

    _faiss_cache[key] = (index, metadata, config)
    return index, metadata, config


def rag_search(query: str, board: str, cls: str, subject: str, k: int = 3) -> list[str]:
    index, metadata, _ = load_faiss(board, cls, subject)
    if index is None or metadata is None or metadata.empty:
        return []

    q_vec = EMBED_MODEL.encode([query]).astype(np.float32)
    _, idxs = index.search(q_vec, k)

    chunks = []
    for i in idxs[0]:
        if i < 0 or i >= len(metadata):
            continue
        row  = metadata.iloc[i]
        text = row.get("content", row.get("text", str(row.to_dict())))
        chunks.append(str(text))
    return chunks


def _preload_all() -> None:
    print("\n" + "="*60)
    print("  📚 PRELOADING CURRICULUM CACHE …")
    print("="*60)

    boards: list[str] = []
    if os.path.exists(CURRICULUM_DIR):
        boards = sorted(
            d for d in os.listdir(CURRICULUM_DIR)
            if os.path.isdir(os.path.join(CURRICULUM_DIR, d))
        )
    _curriculum_cache["boards"] = boards

    total_subjects = 0
    for board in boards:
        board_path = os.path.join(CURRICULUM_DIR, board)
        classes = sorted(
            d for d in os.listdir(board_path)
            if os.path.isdir(os.path.join(board_path, d))
        )
        _curriculum_cache["classes"][board] = classes
        print(f"  🏫 {board}: {len(classes)} class(es)")

        for cls in classes:
            cls_path = os.path.join(board_path, cls)
            subjects = sorted(
                d for d in os.listdir(cls_path)
                if os.path.isdir(os.path.join(cls_path, d))
            )
            _curriculum_cache["subjects"][f"{board}/{cls}"] = subjects

            for subj in subjects:
                json_path = os.path.join(cls_path, subj, "curriculum.json")
                if os.path.exists(json_path):
                    with open(json_path, encoding="utf-8") as f:
                        topics = json.load(f)
                else:
                    topics = []
                _curriculum_cache["curriculum"][f"{board}/{cls}/{subj}"] = topics
                total_subjects += 1

    print(f"\n  ✅ Curriculum cache ready: "
          f"{len(boards)} board(s), "
          f"{sum(len(v) for v in _curriculum_cache['classes'].values())} class combo(s), "
          f"{total_subjects} subject(s) cached.\n")

    print("="*60)
    print("  🧠 PRELOADING ALL FAISS KNOWLEDGE-BASE INDICES …")
    print("="*60)

    faiss_loaded = 0
    faiss_missing = 0
    for board in boards:
        for cls in _curriculum_cache["classes"].get(board, []):
            for subj in _curriculum_cache["subjects"].get(f"{board}/{cls}", []):
                idx, meta, cfg = load_faiss(board, cls, subj)
                if idx is not None:
                    faiss_loaded += 1
                    n_vecs = idx.ntotal
                    print(f"  ✅ {board}/{cls}/{subj}  ({n_vecs} vectors)")
                else:
                    faiss_missing += 1
                    print(f"  ⚠️  {board}/{cls}/{subj}  — no FAISS index found, skipping")

    print(f"\n  🏆 FAISS preload complete: "
          f"{faiss_loaded} index(es) loaded, {faiss_missing} skipped.\n")
    print("="*60)
    print("  🚀 ALL CACHES WARM — MARIA is ready to teach instantly! 🚀")
    print("="*60 + "\n")


_preload_all()


# ── Qwen inference ────────────────────────────────────────────────────────────

def _build_text(messages: list[dict], enable_thinking: bool = True) -> str:
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )


def call_model(
    messages: list[dict],
    system:   str  = "",
    max_new_tokens: int  = 512,
    enable_thinking: bool = True,
    temperature: float   = 0.7,
) -> str:
    if system:
        messages = [{"role": "system", "content": system}] + messages

    text   = _build_text(messages, enable_thinking)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)


def parse_thinking(raw: str) -> tuple[str, str]:
    m       = re.search(r"<think>(.*?)</think>", raw, re.DOTALL)
    thinking = m.group(1).strip() if m else ""
    response = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()
    return thinking, response


# ── Decision making ───────────────────────────────────────────────────────────

DECISIONS = ("BLOCK", "CURRICULUM", "QUESTION", "CHITCHAT")

def make_decision(
    message:              str,
    chat_history:         list[dict],
    scratchpad:           str,
    current_learning:     dict | None,
    curriculum_objectives: list[str],
) -> str:
    obj_text     = "\n".join(f"  • {o}" for o in curriculum_objectives[:12])
    current_text = json.dumps(current_learning, ensure_ascii=False) if current_learning else "None"
    hist_text    = "\n".join(f"{m['role']}: {m['content'][:80]}" for m in chat_history[-6:])

    prompt = f"""You are a classifier for a children's education app (ages 5-12).

Classify the student's message into EXACTLY ONE category:

  BLOCK      → Contains inappropriate, abusive, or adult content.
               Also choose BLOCK if chat_history shows persistent disrespect.
  CURRICULUM → Directly relates to the CURRENT learning objective below.
  QUESTION   → Educational, but about a DIFFERENT topic / lesson / subject.
  CHITCHAT   → Off-topic, friendly talk, curiosity about you, etc.

Evaluation order:
  1. Inappropriate? → BLOCK
  2. About current_learning? → CURRICULUM
  3. Educational but different topic? → QUESTION
  4. None of the above? → CHITCHAT

──────────────────────────────
Current Learning:
{current_text}

All Curriculum Objectives:
{obj_text}

Recent Chat:
{hist_text}

Scratchpad:
{scratchpad[-500:] if scratchpad else "(empty)"}
──────────────────────────────
Student Message: "{message}"

Reply with ONLY ONE WORD (BLOCK / CURRICULUM / QUESTION / CHITCHAT):"""

    raw = call_model(
        [{"role": "user", "content": prompt}],
        max_new_tokens=10,
        enable_thinking=False,
        temperature=0.1,
    )
    _, answer = parse_thinking(raw)
    answer    = answer.strip().upper()
    for d in DECISIONS:
        if d in answer:
            return d
    return "CHITCHAT"


# ── Response generation ───────────────────────────────────────────────────────

DECISION_SYSTEM_TEMPLATES = {
    "BLOCK": """{persona}
You are MARIA, a kind teacher for children ages 5-12.
The student typed something inappropriate. Gently redirect them — no shame, no anger.
Use 1-2 friendly sentences and bring them back to learning. Emojis welcome 😊.""",

    "CURRICULUM": """{persona}
You are MARIA, an enthusiastic teacher for children ages 5-12. 🌟
Current lesson: {current}
Knowledge-base context: {context}
Chat so far: {history}
Scratchpad (your private notes): {scratchpad}

RULES:
• Answer warmly, clearly, with LOTS of encouraging emojis 🎉
• Use simple words a 6-12 year old understands
• Give a VISUAL EXPLANATION — paint a vivid picture with analogies, comparisons,
  little stories, and descriptive imagery so the child can SEE the idea 🌈
• After answering, gently check if the student understood or nudge to next topic
• NEVER say anything harsh, adult, or inappropriate
• At the very end of your response add this line (replace <...> with your content):
  [VIDEO_PROMPT]: <a short animated video scene showing what you just explained, with motion and movement, suitable for kids age 6-12, bright colours, cartoon style>""",

    "QUESTION": """{persona}
You are MARIA, a warm and knowledgeable teacher for children ages 5-12. 📚
The student asked a question OUTSIDE the current lesson — still great curiosity!
Relevant knowledge: {context}
Current lesson: {current}

RULES:
• Answer briefly and encouragingly, with emojis 😊
• Give a VISUAL EXPLANATION — use analogies, vivid imagery, and descriptive comparisons
  so the child can picture exactly what you mean 🖼️
• After answering, gently redirect back to the current lesson
• Keep language simple and age-appropriate
• At the very end of your response add this line (replace <...> with your content):
  [VIDEO_PROMPT]: <a short animated video scene showing what you just explained, with motion and movement, suitable for kids age 6-12, bright colours, cartoon style>""",

    "CHITCHAT": """{persona}
You are MARIA, a warm, playful friend-teacher for children ages 5-12. 🦄
Current lesson: {current}
Chat so far: {history}

RULES:
• Be friendly, warm, curious — match the child's energy 🌟
• Use emojis liberally 😄🎈
• After 1-2 friendly lines, gently guide back to learning
• NEVER be dismissive or harsh""",
}


def respond(
    decision:        str,
    message:         str,
    chat_history:    list[dict],
    scratchpad:      str,
    persona_text:    str,
    current_learning: dict | None,
    board:           str,
    cls:             str,
    subject:         str,
) -> tuple[str, str, str, str | None]:
    hist_text    = "\n".join(f"{m['role']}: {m['content'][:100]}" for m in chat_history[-8:])
    current_text = json.dumps(current_learning, ensure_ascii=False) if current_learning else "None"

    context = ""
    if decision in ("CURRICULUM", "QUESTION"):
        chunks  = rag_search(message, board, cls, subject, k=3)
        context = "\n\n".join(chunks) if chunks else "Use your general knowledge."

    tpl = DECISION_SYSTEM_TEMPLATES[decision]
    system = tpl.format(
        persona=persona_text,
        current=current_text,
        context=context,
        history=hist_text,
        scratchpad=scratchpad[-600:],
    )

    user_content = message if decision != "BLOCK" else (
        f"The student said something inappropriate: '{message}'. "
        "Redirect gently in 1-2 sentences with kindness and emojis."
    )

    raw      = call_model(
        [{"role": "user", "content": user_content}],
        system=system,
        max_new_tokens=500,
        enable_thinking=True,
    )
    thinking, raw_reply = parse_thinking(raw)

    # ── Extract [VIDEO_PROMPT] for CURRICULUM / QUESTION decisions ────────────
    img_path   = None
    reply      = raw_reply

    if decision in ("CURRICULUM", "QUESTION"):
        img_match = re.search(
            r"\[VIDEO_PROMPT\]\s*:\s*(.+?)(?:\n|$)", raw_reply, re.IGNORECASE | re.DOTALL
        )
        if img_match:
            img_prompt = img_match.group(1).strip()
            # Remove the [VIDEO_PROMPT] line from the displayed reply
            reply = re.sub(
                r"\[VIDEO_PROMPT\]\s*:.*?(?:\n|$)", "", raw_reply,
                flags=re.IGNORECASE | re.DOTALL
            ).strip()
            # Generate the video
            img_path = generate_visual_video(img_prompt)

    entry       = (
        f"[Decision={decision}]\n"
        f"[THINKING]: {thinking[:400]}\n"
        f"[Q]: {message[:60]}\n"
        f"[A]: {reply[:120]}"
    )
    new_scratch = (scratchpad + "\n" + entry)[-2500:]

    return reply, thinking, new_scratch, img_path


# %%
# ╔════════════════════════════════════════════════════════════════════╗
# ║  🎨  CELL 5 — Build & Launch MARIA's Gradio Interface!  🚀       ║
# ╚════════════════════════════════════════════════════════════════════╝

# ── Kids CSS ─────────────────────────────────────────────────────────────────

KIDS_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Nunito:wght@400;600;700;800;900&family=Baloo+2:wght@400;700;800&display=swap');

*, *::before, *::after {
    font-family: 'Nunito', 'Baloo 2', 'Comic Sans MS', cursive, sans-serif !important;
    box-sizing: border-box;
}

/* ── Background ── */
.gradio-container {
    background: linear-gradient(135deg,
        #667eea 0%, #764ba2 15%,
        #f093fb 35%, #f5576c 55%,
        #fda085 70%, #f6d365 85%,
        #fda085 100%) !important;
    background-size: 300% 300% !important;
    animation: gradientShift 12s ease infinite !important;
    min-height: 100vh !important;
}
@keyframes gradientShift {
    0%   { background-position: 0% 50%; }
    50%  { background-position: 100% 50%; }
    100% { background-position: 0% 50%; }
}

/* ── Cards ── */
.card {
    background: rgba(255,255,255,0.96) !important;
    border-radius: 24px !important;
    padding: 24px !important;
    margin: 10px !important;
    box-shadow: 0 12px 40px rgba(0,0,0,0.18) !important;
    border: 3px solid rgba(255,255,255,0.6) !important;
    backdrop-filter: blur(8px) !important;
}

/* ── Header ── */
.maria-header {
    background: rgba(255,255,255,0.97);
    border-radius: 24px;
    padding: 18px 24px;
    margin: 10px 10px 0;
    text-align: center;
    box-shadow: 0 8px 32px rgba(118,75,162,0.25);
    border: 3px solid #ffd700;
    position: relative;
    overflow: hidden;
}
.maria-header::before {
    content: "";
    position: absolute;
    top: 0; left: 0; right: 0; bottom: 0;
    background: linear-gradient(135deg, rgba(255,215,0,0.08), rgba(255,100,150,0.08));
    pointer-events: none;
}

/* ── Buttons ── */
button.primary {
    background: linear-gradient(135deg, #667eea, #764ba2) !important;
    color: white !important;
    border-radius: 50px !important;
    font-size: 1.1em !important;
    font-weight: 800 !important;
    padding: 12px 36px !important;
    border: none !important;
    cursor: pointer !important;
    transition: all 0.3s ease !important;
    box-shadow: 0 6px 20px rgba(102,126,234,0.45) !important;
    letter-spacing: 0.5px !important;
}
button.primary:hover {
    transform: scale(1.06) translateY(-2px) !important;
    box-shadow: 0 10px 30px rgba(102,126,234,0.6) !important;
}
button.secondary {
    background: linear-gradient(135deg, #fda085, #f6d365) !important;
    color: white !important;
    border-radius: 50px !important;
    font-weight: 700 !important;
    border: none !important;
    box-shadow: 0 4px 15px rgba(253,160,133,0.4) !important;
    transition: all 0.3s !important;
}
button.secondary:hover {
    transform: scale(1.04) !important;
}

/* ── Persona cards ── */
.persona-wrap label {
    background: linear-gradient(135deg, #e0c3fc, #8ec5fc) !important;
    border-radius: 16px !important;
    padding: 14px 18px !important;
    margin: 6px !important;
    border: 2.5px solid transparent !important;
    cursor: pointer !important;
    transition: all 0.3s !important;
    font-weight: 700 !important;
    font-size: 0.95em !important;
}
.persona-wrap label:hover {
    transform: scale(1.05) !important;
    border-color: #667eea !important;
    box-shadow: 0 6px 20px rgba(102,126,234,0.35) !important;
}

/* ════════════════════════════════════════════════════════
   TAP-TO-SELECT CARDS  (Board · Class · Subject)
   ════════════════════════════════════════════════════════ */

/* Section label above each group */
.tap-section-label {
    font-size: 0.88em;
    font-weight: 800;
    color: #888;
    letter-spacing: 0.8px;
    text-transform: uppercase;
    margin: 14px 0 8px 4px;
}

/* The wrapping flex row produced by gr.Radio */
.tap-cards .wrap,
.tap-cards > div {
    display: flex !important;
    flex-wrap: wrap !important;
    gap: 10px !important;
    padding: 4px 0 !important;
}

/* Hide the raw radio circle */
.tap-cards input[type="radio"] {
    display: none !important;
}

/* Every option label = a pill card */
.tap-cards label {
    display: inline-flex !important;
    align-items: center !important;
    justify-content: center !important;
    background: linear-gradient(135deg, #f0f4ff, #e8f0fe) !important;
    color: #555 !important;
    border-radius: 50px !important;
    padding: 10px 22px !important;
    border: 2.5px solid #d0d8f0 !important;
    cursor: pointer !important;
    transition: all 0.22s ease !important;
    font-weight: 700 !important;
    font-size: 0.95em !important;
    min-width: 70px !important;
    text-align: center !important;
    box-shadow: 0 3px 10px rgba(102,126,234,0.10) !important;
    user-select: none !important;
}

/* Hover state */
.tap-cards label:hover {
    transform: scale(1.07) translateY(-2px) !important;
    border-color: #667eea !important;
    background: linear-gradient(135deg, #dde4ff, #c8d8ff) !important;
    box-shadow: 0 6px 18px rgba(102,126,234,0.30) !important;
    color: #3a3a8a !important;
}

/* Selected / checked state — uses :has() (well-supported in modern browsers) */
.tap-cards label:has(input[type="radio"]:checked) {
    background: linear-gradient(135deg, #667eea, #764ba2) !important;
    color: white !important;
    border-color: #667eea !important;
    box-shadow: 0 6px 20px rgba(102,126,234,0.50) !important;
    transform: scale(1.07) translateY(-2px) !important;
}

/* Board cards — slightly bigger, bolder */
.tap-cards-board label {
    font-size: 1.05em !important;
    padding: 12px 28px !important;
    border-radius: 50px !important;
}

/* Class cards — numbered feel */
.tap-cards-class label {
    min-width: 60px !important;
    font-size: 1em !important;
}

/* Subject cards — colourful gradient per card via nth-child */
.tap-cards-subject label:nth-child(1)  { background: linear-gradient(135deg,#ffe9ec,#ffd6db) !important; border-color:#ffb3bc !important; }
.tap-cards-subject label:nth-child(2)  { background: linear-gradient(135deg,#fff3e0,#ffe0b2) !important; border-color:#ffcc80 !important; }
.tap-cards-subject label:nth-child(3)  { background: linear-gradient(135deg,#e8f5e9,#c8e6c9) !important; border-color:#a5d6a7 !important; }
.tap-cards-subject label:nth-child(4)  { background: linear-gradient(135deg,#e3f2fd,#bbdefb) !important; border-color:#90caf9 !important; }
.tap-cards-subject label:nth-child(5)  { background: linear-gradient(135deg,#f3e5f5,#e1bee7) !important; border-color:#ce93d8 !important; }
.tap-cards-subject label:nth-child(6)  { background: linear-gradient(135deg,#e0f7fa,#b2ebf2) !important; border-color:#80deea !important; }
.tap-cards-subject label:nth-child(7)  { background: linear-gradient(135deg,#fce4ec,#f8bbd0) !important; border-color:#f48fb1 !important; }
.tap-cards-subject label:nth-child(8)  { background: linear-gradient(135deg,#fffde7,#fff9c4) !important; border-color:#fff176 !important; }
.tap-cards-subject label:nth-child(n+9){ background: linear-gradient(135deg,#f1f8e9,#dcedc8) !important; border-color:#aed581 !important; }

/* Checked state always wins for subject cards */
.tap-cards-subject label:has(input[type="radio"]:checked) {
    background: linear-gradient(135deg, #667eea, #764ba2) !important;
    color: white !important;
    border-color: #667eea !important;
    box-shadow: 0 6px 20px rgba(102,126,234,0.50) !important;
}

/* Divider between sections */
.sel-divider {
    border: none;
    border-top: 2px dashed #e0e0f0;
    margin: 16px 0 4px;
}

/* ── Typing indicator ── */
.typing-box {
    background: linear-gradient(135deg, #a8edea 0%, #fed6e3 100%);
    border-radius: 16px;
    padding: 14px 18px;
    font-style: italic;
    color: #555;
    font-size: 0.95em;
    margin: 8px 0;
    border-left: 5px solid #667eea;
    position: relative;
    overflow: hidden;
    animation: slideIn 0.4s ease;
}
.typing-box::after {
    content: "";
    position: absolute;
    top: 0; left: 0; right: 0; bottom: 0;
    background: linear-gradient(90deg, transparent, rgba(255,255,255,0.3), transparent);
    animation: shimmer 2s infinite;
}
@keyframes slideIn  { from { opacity:0; transform:translateY(8px); } to { opacity:1; transform:translateY(0); } }
@keyframes shimmer  { 0%,100% { transform:translateX(-100%); } 100% { transform:translateX(100%); } }

/* ── Dots animation ── */
.dot { display:inline-block; width:10px; height:10px; border-radius:50%;
       background:#667eea; margin:0 3px; animation:dotBounce 0.7s infinite ease; }
.dot:nth-child(2) { animation-delay:0.15s; background:#f5576c; }
.dot:nth-child(3) { animation-delay:0.3s;  background:#fda085; }
@keyframes dotBounce { 0%,100%{transform:translateY(0)} 50%{transform:translateY(-10px)} }

/* ── Progress bar ── */
.progress-pill {
    background: rgba(255,255,255,0.9);
    border-radius: 50px;
    padding: 8px 16px;
    font-size: 0.85em;
    font-weight: 700;
    color: #555;
    box-shadow: 0 3px 10px rgba(0,0,0,0.12);
    display: inline-block;
}
.progress-bar-outer {
    background: #e8e8f0;
    border-radius: 50px;
    height: 14px;
    overflow: hidden;
    margin: 6px 0;
}
.progress-bar-inner {
    height: 100%;
    border-radius: 50px;
    background: linear-gradient(90deg, #667eea, #f5576c);
    transition: width 0.6s ease;
}

/* ── Chatbot ── */
.message-wrap { border-radius: 18px !important; }

/* ── Decision badge ── */
.badge {
    display: inline-block;
    padding: 3px 12px;
    border-radius: 50px;
    font-size: 0.78em;
    font-weight: 800;
    margin-bottom: 6px;
    text-transform: uppercase;
    letter-spacing: 1px;
}
.badge-BLOCK      { background:#ff6b6b; color:white; }
.badge-CURRICULUM { background:#6bcb77; color:white; }
.badge-QUESTION   { background:#4d96ff; color:white; }
.badge-CHITCHAT   { background:#ffd166; color:#333; }

/* ── Topic pill ── */
.topic-pill {
    background: linear-gradient(135deg, #a8edea, #fed6e3);
    border-radius: 14px;
    padding: 10px 16px;
    font-size: 0.88em;
    font-weight: 700;
    color: #444;
    margin: 4px 0;
    border-left: 4px solid #667eea;
}

/* ── Processing steps ── */
.proc-step {
    font-size: 0.82em;
    color: #555;
    margin: 4px 0;
    padding: 2px 0;
    opacity: 0;
    animation: fadeStep 0.5s ease forwards;
    display: flex;
    align-items: center;
    gap: 6px;
}
@keyframes fadeStep {
    from { opacity:0; transform:translateX(-10px); }
    to   { opacity:1; transform:translateX(0); }
}

footer { display:none !important; }
"""

# ── Confetti helper ───────────────────────────────────────────────────────────

CONFETTI_JS = """
<script>
(function(){
  if(window._confettiLoaded) { triggerConfetti(); return; }
  var s=document.createElement('script');
  s.src='https://cdn.jsdelivr.net/npm/canvas-confetti@1.9.2/dist/confetti.browser.min.js';
  s.onload=function(){ window._confettiLoaded=true; triggerConfetti(); };
  document.head.appendChild(s);
  function triggerConfetti(){
    confetti({ particleCount:180, spread:90, origin:{y:0.55},
               colors:['#667eea','#f5576c','#fda085','#6bcb77','#ffd166'] });
    setTimeout(()=>confetti({ particleCount:80, spread:120, origin:{y:0.7} }), 400);
  }
})();
</script>
"""


# ── Build Gradio App ──────────────────────────────────────────────────────────

def build_maria():
    persona_map: dict[str, str] = {}
    personas_list: list[str]    = []
    if TEACHER_PERSONAS and "teacher_personas" in TEACHER_PERSONAS:
        for p in TEACHER_PERSONAS["teacher_personas"]:
            persona_map[p["name"]] = p["persona"]
            personas_list.append(p["name"])

    # ─── Initial state template ───────────────────────────────────────────────
    def fresh_state() -> dict:
        return dict(
            persona_name  = "",
            persona_text  = "",
            board         = "",
            cls           = "",
            subject       = "",
            all_topics    = [],
            sel_topics    = [],
            topic_idx     = 0,
            typing_idx    = 0,
            current       = None,
            chat_history  = [],
            scratchpad    = "",
        )

    # ─── Gradio Blocks ────────────────────────────────────────────────────────
    with gr.Blocks(css=KIDS_CSS, title="🌈 MARIA — Your AI Teacher!") as demo:

        state = gr.State(fresh_state())

        # ════════════════════════════════════
        # HEADER
        # ════════════════════════════════════
        gr.HTML("""
        <div class="maria-header">
          <div style="font-size:3em;line-height:1;margin-bottom:4px;">🌈</div>
          <h1 style="font-size:2.6em;margin:0;font-family:'Baloo 2',cursive;
                     background:linear-gradient(135deg,#667eea,#f5576c,#fda085);
                     -webkit-background-clip:text;-webkit-text-fill-color:transparent;
                     font-weight:900;letter-spacing:-1px;">
            M · A · R · I · A
          </h1>
          <p style="margin:4px 0 0;color:#777;font-size:1.05em;font-weight:700;">
            ✨ Magical AI Reading &amp; Instruction Assistant ✨
          </p>
          <div style="font-size:1.6em;margin-top:8px;letter-spacing:4px;">
            🦋 🌟 📚 🎨 🎵 🦄 🌈 ⭐ 🎉 🐝
          </div>
        </div>
        """)

        # ════════════════════════════════════
        # SCREEN 1 — Persona selection
        # ════════════════════════════════════
        with gr.Column(visible=True, elem_classes=["card"]) as scr1:
            gr.HTML("""
            <div style="text-align:center;padding:8px 0 16px;">
              <h2 style="color:#667eea;font-size:1.75em;margin:0;">
                👩‍🏫 Pick Your Teacher Today! 👨‍🏫
              </h2>
              <p style="color:#999;font-size:1em;margin:6px 0 0;">
                Choose the teaching style that sounds most exciting! 🎉
              </p>
            </div>
            """)

            persona_radio = gr.Radio(
                choices  = personas_list,
                value    = personas_list[0] if personas_list else None,
                label    = "🌟 Who do you want to learn with today?",
                elem_classes=["persona-wrap"],
            )

            persona_desc = gr.HTML(
                f'<div class="topic-pill">📖 <b>About:</b> '
                f'{persona_map.get(personas_list[0], "") if personas_list else ""}</div>'
            )

            def _update_desc(name):
                desc = persona_map.get(name, "")
                return f'<div class="topic-pill">📖 <b>About:</b> {desc}</div>'
            persona_radio.change(_update_desc, [persona_radio], [persona_desc])

            s1_err     = gr.HTML("")
            s1_next    = gr.Button("Next ➡️ Choose Your Lessons!", variant="primary")

        # ════════════════════════════════════
        # SCREEN 2 — Curriculum selection
        # (Board / Class / Subject as tap cards)
        # ════════════════════════════════════
        with gr.Column(visible=False, elem_classes=["card"]) as scr2:
            gr.HTML("""
            <div style="text-align:center;padding:8px 0 16px;">
              <h2 style="color:#f5576c;font-size:1.75em;margin:0;">
                📚 Choose Your Lessons! 📚
              </h2>
              <p style="color:#999;font-size:1em;margin:6px 0 0;">
                Tap your board, class &amp; subject — then choose which topics! 🎯
              </p>
            </div>
            """)

            # ── Board tap-cards ───────────────────────────────────────────────
            gr.HTML('<div class="tap-section-label">🏫 Board</div>')
            rd_board = gr.Radio(
                choices      = get_boards(),
                value        = None,
                label        = "",
                show_label   = False,
                interactive  = True,
                elem_classes = ["tap-cards", "tap-cards-board"],
            )

            # ── Class tap-cards ───────────────────────────────────────────────
            gr.HTML('<hr class="sel-divider"><div class="tap-section-label">🎒 Class</div>')
            rd_class = gr.Radio(
                choices      = [],
                value        = None,
                label        = "",
                show_label   = False,
                interactive  = True,
                elem_classes = ["tap-cards", "tap-cards-class"],
            )

            # ── Subject tap-cards ─────────────────────────────────────────────
            gr.HTML('<hr class="sel-divider"><div class="tap-section-label">📝 Subject</div>')
            rd_subject = gr.Radio(
                choices      = [],
                value        = None,
                label        = "",
                show_label   = False,
                interactive  = True,
                elem_classes = ["tap-cards", "tap-cards-subject"],
            )

            gr.HTML('<hr style="border:2px dashed #ffd700;margin:18px 0 14px;">')
            gr.HTML("""
            <div style="text-align:center;margin:8px 0;">
              <h3 style="color:#667eea;margin:0;">📋 Topics We'll Learn!</h3>
              <p style="color:#999;font-size:0.9em;margin:4px 0 0;">
                Tick or untick to choose exactly what you want 🙌
              </p>
            </div>
            """)

            cb_all    = gr.Checkbox(label="✅ Select / Deselect All", value=True)
            cb_topics = gr.CheckboxGroup(choices=[], value=[], label="", interactive=True)
            topic_preview = gr.HTML("")

            store_curriculum = gr.State([])

            # ── Cascade handlers ──────────────────────────────────────────────

            def _on_board(board):
                cls_choices = get_classes(board) if board else []
                return (
                    gr.Radio(choices=cls_choices, value=None),
                    gr.Radio(choices=[], value=None),
                    gr.CheckboxGroup(choices=[], value=[]),
                    gr.HTML(""),
                    [],
                )

            def _on_class(board, cls):
                subj_choices = get_subjects(board, cls) if (board and cls) else []
                return (
                    gr.Radio(choices=subj_choices, value=None),
                    gr.CheckboxGroup(choices=[], value=[]),
                    gr.HTML(""),
                    [],
                )

            def _on_subject(board, cls, subject):
                if not (board and cls and subject):
                    return gr.CheckboxGroup(choices=[], value=[]), gr.HTML(""), []
                topics = get_curriculum(board, cls, subject)
                if not topics:
                    return gr.CheckboxGroup(choices=[], value=[]), gr.HTML(""), []
                labels = [f"📖 {t['topic']}" for t in topics]
                preview_html = "".join(
                    f"""<div class="topic-pill" style="margin:6px 0;">
                          <b>📌 {t['topic']}</b><br>
                          <small style="color:#777;">{t['content'][:130]}…</small>
                        </div>"""
                    for t in topics[:4]
                )
                return gr.CheckboxGroup(choices=labels, value=labels), gr.HTML(preview_html), topics

            def _on_select_all(checked, board, cls, subject):
                topics = get_curriculum(board, cls, subject) if (board and cls and subject) else []
                labels = [f"📖 {t['topic']}" for t in topics]
                return gr.CheckboxGroup(choices=labels, value=labels if checked else [])

            rd_board.change(
                _on_board,
                [rd_board],
                [rd_class, rd_subject, cb_topics, topic_preview, store_curriculum],
            )
            rd_class.change(
                _on_class,
                [rd_board, rd_class],
                [rd_subject, cb_topics, topic_preview, store_curriculum],
            )
            rd_subject.change(
                _on_subject,
                [rd_board, rd_class, rd_subject],
                [cb_topics, topic_preview, store_curriculum],
            )
            cb_all.change(
                _on_select_all,
                [cb_all, rd_board, rd_class, rd_subject],
                [cb_topics],
            )

            with gr.Row():
                s2_back = gr.Button("⬅️ Back", variant="secondary")
                s2_next = gr.Button("Next ➡️ Start Learning! 🚀", variant="primary")

        # ════════════════════════════════════
        # SCREEN 3 — Chatbot
        # ════════════════════════════════════
        with gr.Column(visible=False, elem_classes=["card"]) as scr3:
            with gr.Row():
                with gr.Column(scale=4):
                    gr.HTML("""
                    <h2 style="color:#764ba2;font-size:1.7em;margin:0;text-align:center;">
                      💬 Chat with MARIA! 💬
                    </h2>
                    <p style="color:#aaa;text-align:center;font-size:0.9em;margin:4px 0 0;">
                      Ask anything — I'm here to help! 🌟
                    </p>
                    """)
                with gr.Column(scale=2):
                    topic_display    = gr.HTML("")
                with gr.Column(scale=2):
                    progress_display = gr.HTML("")

            chatbot = gr.Chatbot(
                label="🌟 MARIA",
                height=440,
                bubble_full_width=False,
                type="messages",
                show_label=True,
                avatar_images=(
                    "https://api.dicebear.com/9.x/bottts-neutral/svg?seed=kid123&backgroundColor=b6e3f4",
                    "https://api.dicebear.com/9.x/bottts-neutral/svg?seed=maria42&backgroundColor=d1d4f9",
                ),
            )

            typing_html = gr.HTML("")
            confetti_area = gr.HTML("")

            # ── Video display (replaces gr.Image) ─────────────────────────────
            image_display = gr.Video(
                label="🎬 MARIA's Visual Explanation",
                visible=False,
                height=380,
                show_download_button=True,
                show_label=True,
                elem_id="maria-visual-video",
            )

            with gr.Row():
                msg_box  = gr.Textbox(
                    placeholder="💬 Type your question here… Don't be shy! 🌟",
                    label="",
                    scale=6,
                    lines=1,
                    max_lines=4,
                    show_label=False,
                )
                send_btn = gr.Button("Send 🚀", scale=1, variant="primary")

            with gr.Row():
                nxt_topic_btn = gr.Button("⏭️ Next Topic!",     variant="secondary")
                back_s3_btn   = gr.Button("⬅️ Back to Lessons", variant="secondary")
                clear_btn     = gr.Button("🗑️ Clear Chat",       variant="secondary")

        # ════════════════════════════════════
        # NAVIGATION — Screen 1 → 2
        # ════════════════════════════════════
        def nav_to_s2(persona_name, st):
            if not persona_name:
                err = '<p style="color:red;font-weight:700;">⚠️ Please pick a teacher first!</p>'
                return gr.update(visible=True), gr.update(visible=False), gr.update(visible=False), gr.HTML(err), st
            st = fresh_state()
            st["persona_name"] = persona_name
            st["persona_text"] = persona_map.get(persona_name, "You are a kind, friendly teacher.")
            return gr.update(visible=False), gr.update(visible=True), gr.update(visible=False), gr.HTML(""), st

        s1_next.click(nav_to_s2, [persona_radio, state],
                      [scr1, scr2, scr3, s1_err, state])

        # Screen 2 → 1
        s2_back.click(
            lambda st: (gr.update(visible=True), gr.update(visible=False), gr.update(visible=False), st),
            [state], [scr1, scr2, scr3, state]
        )

        # Screen 3 → 2
        back_s3_btn.click(
            lambda st: (gr.update(visible=False), gr.update(visible=True), gr.update(visible=False), st),
            [state], [scr1, scr2, scr3, state]
        )

        clear_btn.click(lambda: ([], gr.HTML("")), [], [chatbot, typing_html])

        # ════════════════════════════════════
        # NAVIGATION — Screen 2 → 3
        # ════════════════════════════════════
        def nav_to_s3(board, cls, subject, sel_labels, curriculum_data, st):
            if not board or not cls or not subject:
                return (gr.update(visible=False), gr.update(visible=True), gr.update(visible=False),
                        [], gr.HTML(""), gr.HTML(""), gr.HTML(""), st)
            if not sel_labels:
                return (gr.update(visible=False), gr.update(visible=True), gr.update(visible=False),
                        [], gr.HTML(""), gr.HTML(""), gr.HTML(""), st)

            selected_names = {lbl.replace("📖 ", "") for lbl in sel_labels}
            sel_topics     = [t for t in curriculum_data if t["topic"] in selected_names]
            if not sel_topics:
                return (gr.update(visible=False), gr.update(visible=True), gr.update(visible=False),
                        [], gr.HTML(""), gr.HTML(""), gr.HTML(""), st)

            st = st.copy()
            st.update(board=board, cls=cls, subject=subject,
                      all_topics=curriculum_data, sel_topics=sel_topics,
                      topic_idx=0, current=sel_topics[0],
                      chat_history=[], scratchpad="")

            topic   = sel_topics[0]
            objs    = "\n".join(f"  ✅ {o}" for o in topic.get("learning_objectives", []))
            welcome = (
                f"🌟 **Hello, Superstar Learner!** 🌟\n\n"
                f"I'm **MARIA**, your magical teacher today! 🦄\n\n"
                f"Today we'll explore:\n📌 **{topic['topic']}**\n\n"
                f"{topic.get('content','')[:220]}…\n\n"
                f"🎯 **Our Goals:**\n{objs}\n\n"
                f"Ready? Let's make learning AMAZING! 🚀🎉"
            )
            history  = [{"role": "assistant", "content": welcome}]
            st["chat_history"] = history

            total    = len(sel_topics)
            prog_html = _progress_html(1, total)
            t_html    = _topic_html(topic["topic"])

            return (gr.update(visible=False), gr.update(visible=False), gr.update(visible=True),
                    history, gr.HTML(t_html), gr.HTML(prog_html), gr.HTML(""), st)

        s2_next.click(
            nav_to_s3,
            [rd_board, rd_class, rd_subject, cb_topics, store_curriculum, state],
            [scr1, scr2, scr3, chatbot, topic_display, progress_display, confetti_area, state]
        )

        # ════════════════════════════════════
        # HELPERS for topic/progress display
        # ════════════════════════════════════
        def _progress_html(current_n: int, total: int) -> str:
            pct  = int(current_n / total * 100)
            stars = "⭐" * current_n + "☆" * (total - current_n)
            return (
                f'<div class="progress-pill">📊 {current_n}/{total}<br>'
                f'<div class="progress-bar-outer">'
                f'<div class="progress-bar-inner" style="width:{pct}%"></div></div>'
                f'<small>{stars}</small></div>'
            )

        def _topic_html(name: str) -> str:
            return (
                f'<div class="topic-pill" style="font-size:0.82em;">'
                f'📌 <b>Now:</b><br>{name[:55]}{"…" if len(name)>55 else ""}</div>'
            )

        # ════════════════════════════════════
        # CHAT (generator for streaming feel)
        # ════════════════════════════════════
        def chat_fn(message: str, history: list, st: dict):
            if not message.strip():
                yield history, gr.HTML(""), gr.HTML(""), st, gr.Video(visible=False)
                return

            history = history + [{"role": "user", "content": message}]

            typing_idx      = st.get("typing_idx", 0)
            next_typing_idx = (typing_idx + 1) % len(TYPING_MESSAGES)

            import json as _json
            msgs_js = _json.dumps(TYPING_MESSAGES)

            processing_steps = [
                ("🔍", "Classifying your question…"),
                ("📚", "Proctoring curriculum alignment…"),
                ("🗂️", "Curating relevant knowledge…"),
                ("🧩", "Matching learning objectives…"),
                ("✍️", "Generating teacher response…"),
                ("✨", "Adding MARIA magic…"),
            ]
            steps_html = "".join(
                f'<div class="proc-step" style="animation-delay:{i * 0.55}s;">'
                f'<span style="font-size:1em;">{icon}</span> {label}'
                f'</div>'
                for i, (icon, label) in enumerate(processing_steps)
            )

            t_html = f"""
            <div class="typing-box">
              <span style="font-size:1.2em;">⏳</span>
              <i id="maria-typing-msg">{TYPING_MESSAGES[typing_idx]}</i>
              <br><br>
              <span class="dot"></span><span class="dot"></span><span class="dot"></span>
              <br>
              <div style="margin-top:10px;padding-left:4px;border-top:1px dashed #ccc;padding-top:8px;">
                <small style="color:#888;font-weight:700;letter-spacing:0.5px;">
                  ⚙️ Qwen3 is working…
                </small>
                {steps_html}
              </div>
            </div>
            <script>
            (function(){{
              const msgs = {msgs_js};
              let idx = {(typing_idx + 1) % len(TYPING_MESSAGES)};
              const el = document.getElementById('maria-typing-msg');
              function rotate(){{
                if (el) {{ el.innerHTML = msgs[idx % msgs.length]; idx++; }}
                setTimeout(rotate, 5000);
              }}
              setTimeout(rotate, 5000);
            }})();
            </script>
            """

            st = st.copy()
            st["typing_idx"] = next_typing_idx
            yield history, gr.HTML(t_html), gr.HTML(""), st, gr.Video(visible=False)

            persona_text     = st.get("persona_text", "You are a kind teacher.")
            current_learning = st.get("current", None)
            sel_topics       = st.get("sel_topics", [])
            board            = st.get("board", "")
            cls              = st.get("cls", "")
            subject          = st.get("subject", "")
            scratchpad       = st.get("scratchpad", "")

            all_objectives = [o for t in sel_topics for o in t.get("learning_objectives", [])]

            model_history = [{"role": m["role"], "content": m["content"]}
                             for m in history[-10:]]

            decision = make_decision(
                message, model_history, scratchpad,
                current_learning, all_objectives
            )

            # ── For visual decisions show ZeroScope V2 progress in typing box ──
            if decision in ("CURRICULUM", "QUESTION"):
                sd_msgs_js = _json.dumps(SD_TYPING_MESSAGES)
                sd_steps = [
                    ("🎬", "Preparing visual explanation…"),
                    ("⚡", "ZeroScope V2: generating short video clip…"),
                    ("🎨", "Rendering colourful learning frames…"),
                    ("🌈", "Composing kid-safe animation…"),
                    ("✅", "Visual video almost ready!"),
                ]
                sd_steps_html = "".join(
                    f'<div class="proc-step" style="animation-delay:{i * 0.6}s;">'
                    f'<span style="font-size:1em;">{icon}</span> {label}'
                    f'</div>'
                    for i, (icon, label) in enumerate(sd_steps)
                )
                sd_t_html = f"""
                <div class="typing-box" style="border-left-color:#f5576c;">
                  <span style="font-size:1.2em;">🎬</span>
                  <i id="maria-sd-msg">{SD_TYPING_MESSAGES[0]}</i>
                  <br><br>
                  <span class="dot"></span><span class="dot"></span><span class="dot"></span>
                  <br>
                  <div style="margin-top:10px;padding-left:4px;border-top:1px dashed #ccc;padding-top:8px;">
                    <small style="color:#888;font-weight:700;letter-spacing:0.5px;">
                      🎬 ZeroScope V2 generating visual video…
                    </small>
                    {sd_steps_html}
                  </div>
                </div>
                <script>
                (function(){{
                  const msgs = {sd_msgs_js};
                  let idx = 1;
                  const el = document.getElementById('maria-sd-msg');
                  function rotate(){{
                    if (el) {{ el.innerHTML = msgs[idx % msgs.length]; idx++; }}
                    setTimeout(rotate, 3000);
                  }}
                  setTimeout(rotate, 3000);
                }})();
                </script>
                """
                yield history, gr.HTML(sd_t_html), gr.HTML(""), st, gr.Video(visible=False)

            reply, thinking, new_scratch, img_path = respond(
                decision, message, model_history, scratchpad,
                persona_text, current_learning, board, cls, subject,
            )

            history  = history + [{"role": "assistant", "content": reply}]

            new_st = st.copy()
            new_st["scratchpad"]   = new_scratch
            new_st["chat_history"] = [{"role": m["role"], "content": m["content"]}
                                       for m in history]

            # ── Yield final response + video (if any) ─────────────────────────
            if img_path:
                yield history, gr.HTML(""), gr.HTML(""), new_st, gr.Video(value=img_path, visible=True)
            else:
                yield history, gr.HTML(""), gr.HTML(""), new_st, gr.Video(visible=False)

        msg_box.submit( chat_fn, [msg_box, chatbot, state], [chatbot, typing_html, confetti_area, state, image_display]).then(lambda: "", [], [msg_box])
        send_btn.click( chat_fn, [msg_box, chatbot, state], [chatbot, typing_html, confetti_area, state, image_display]).then(lambda: "", [], [msg_box])

        # ════════════════════════════════════
        # NEXT TOPIC
        # ════════════════════════════════════
        def next_topic_fn(history: list, st: dict):
            sel_topics = st.get("sel_topics", [])
            idx        = st.get("topic_idx", 0) + 1
            total      = len(sel_topics)

            if idx >= total:
                finish = (
                    "🎊🎉 **AMAZING WORK, SUPERSTAR!** 🎉🎊\n\n"
                    "You've completed ALL the topics for today!\n\n"
                    "🌟 You are absolutely **BRILLIANT**! 🌟\n"
                    "🦄 MARIA is SO incredibly proud of you! 🦄\n"
                    "🏆 You're a true **Learning Champion**! 🏆\n\n"
                    "Come back tomorrow for more magical adventures! 🚀📚✨"
                )
                history = history + [{"role": "assistant", "content": finish}]
                prog_html  = _progress_html(total, total)
                topic_html = '<div class="topic-pill">🏆 All Topics Done! 🏆</div>'
                new_st     = st.copy()
                new_st["topic_idx"] = idx
                return history, gr.HTML(topic_html), gr.HTML(prog_html), gr.HTML(CONFETTI_JS), new_st

            topic  = sel_topics[idx]
            objs   = "\n".join(f"  ✅ {o}" for o in topic.get("learning_objectives", []))
            msg    = (
                f"🎯 **Great job! Moving to the next topic!** 🎯\n\n"
                f"📌 **{topic['topic']}**\n\n"
                f"{topic.get('content','')[:200]}…\n\n"
                f"🎯 **New Goals:**\n{objs}\n\n"
                f"Ready? Let's go! 🚀"
            )
            history  = history + [{"role": "assistant", "content": msg}]
            new_st   = st.copy()
            new_st.update(topic_idx=idx, current=topic, scratchpad="")

            prog_html  = _progress_html(idx + 1, total)
            topic_html = _topic_html(topic["topic"])
            return history, gr.HTML(topic_html), gr.HTML(prog_html), gr.HTML(""), new_st

        nxt_topic_btn.click(
            next_topic_fn,
            [chatbot, state],
            [chatbot, topic_display, progress_display, confetti_area, state]
        )

    return demo


# %%
# ╔════════════════════════════════════════════════════════╗
# ║  🚀  CELL 6 — Launch MARIA! Open the Gradio link! 🌈  ║
# ╚════════════════════════════════════════════════════════╝

print("""
🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈
  🚀 Launching MARIA — Your Magical AI Teacher! 🚀
  Look for the 🔗 public URL below and open it!
🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈🌈
""")

demo = build_maria()
demo.queue().launch(
    share      = True,
    debug      = False,
    show_error = True,
    server_name= "0.0.0.0",
)